# Traditional OCR vs. Vision-Language OCR — a CEFR Text-Extraction Benchmark

**PaddleOCR (PP-OCRv5)** vs **PaddleOCR-VL**, plus a controlled fine-tuning study on **TRDG / SynthText / IAM**.

Runs **start to finish on Kaggle**. Turn on **GPU** (*Settings → Accelerator → GPU T4*) and **Internet** (*Settings → Internet → On*).

---
### What this notebook decides
The goal is to pick the **best OCR approach for extracting clean printed English text** that will later feed a **CEFR regression** model. The decisive metric is **CER on clean printed text** (the target domain); **WER, inference time, memory** are secondary.

### What is real, what is a proxy (read this — it keeps the results honest)
- **Two pre-trained anchors are the real models:** `PP-OCRv5` (via `paddleocr`) and `PaddleOCR-VL` 0.9B (via `transformers`). They are run **pre-trained**, never faked.
- **The six-way fine-tuning study runs on a trainable CRNN+CTC recognizer** — a transparent, reproducible stand-in for the *recognition stage* of a traditional OCR pipeline. PaddleOCR-VL has **no released fine-tuning workflow** (the official ERNIEKit recipe is "coming soon"), so fine-tuning a 0.9B VLM six ways inside one Kaggle session cannot be done honestly here. Appendix A gives the real PP-OCRv5 fine-tuning recipe; Appendix B gives a VL LoRA scaffold.
- **Datasets are real-if-attached, else clearly-labeled synthetic proxies** (full SynthText is ~40 GB; IAM needs registration). Every result row carries a `source` tag. Appendix D shows how to plug in the real data.

### Why this version is trustworthy (the fixes that matter)
Earlier CRNN benchmarks of this kind silently fail: with no LR warmup and a too-small net, CTC training collapses to all-blank, every model ties at CER≈1.0, and the "ranking" is noise. This notebook fixes that and **guarantees the numbers mean something**:
1. **Fully reproducible** — `torch`/CUDA/cuDNN seeded and `DataLoader` seeded, so re-running gives the *same* result.
2. **The recognizer actually learns** — LR **warmup**, canonical CNN widths, **aspect-preserving** preprocessing, blank-bias init.
3. **Best-model selection** — training restores the **best-validation-CER epoch** (not the last), and the best is saved for download.
4. **Convergence sanity-gate** — if the recognizer did not learn, the notebook prints a loud warning so you never trust a bad run.
5. **Dataset verification** — an explicit integrity check (empty labels, charset coverage, image sizes, train/test leakage) and a clear report of what is real vs proxy.


## 0. Configuration

All budget knobs live here. Defaults are tuned to **converge on a Kaggle GPU**. On CPU they will be slow — keep GPU on.

In [ ]:
import os, math, time, json, random, string, gc
import numpy as np

# ---- Reproducibility: a single switch that seeds EVERYTHING ----
SEED = 42
def set_seed(seed=SEED):
    """Seed python, numpy and torch (incl. CUDA) and force deterministic kernels.
    Called before data generation and before every training run, so results do
    NOT change from run to run (this is the fix for 'every train changes results')."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try: torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception: pass
    except Exception:
        pass
set_seed()

CFG = dict(
    # ---- what to run ----
    RUN_PRETRAINED_PADDLEOCR = True,
    RUN_PRETRAINED_VL        = True,    # set False to skip the VLM (saves time / VRAM)
    RUN_FINETUNE_EXPERIMENTS = True,
    INCLUDE_IAM_EXPERIMENT   = True,    # experiments 4 and 6 (optional)

    # ---- dataset sizes (must be big enough for the CRNN to learn) ----
    N_BASE_TRAIN = 4000,   # base printed corpus -> the "pre-trained" CRNN
    N_FT_TRAIN   = 2000,   # samples per fine-tuning dataset
    N_VAL        = 400,
    N_TEST       = 400,    # held-out CLEAN PRINTED test set = the CEFR target domain

    # ---- recognizer / training (warmup is the key to convergence) ----
    IMG_H = 32, IMG_W = 200,
    EPOCHS_BASE = 40, EPOCHS_FT = 18,
    WARMUP_EPOCHS = 6,            # linear LR warmup; prevents the all-blank CTC collapse
    BATCH = 128, LR = 8e-4,
    SANITY_CER_THRESHOLD = 0.60,  # base val_CER above this => results flagged untrustworthy

    # ---- pretrained models ----
    PADDLE_DEVICE    = "cpu",                  # 'cpu' is most reliable on Kaggle; 'gpu' if paddle-gpu installs cleanly
    PADDLE_REC_MODEL = "PP-OCRv5_mobile_rec",  # 'PP-OCRv5_server_rec' is more accurate but slower
    VL_MODEL_ID      = "PaddlePaddle/PaddleOCR-VL",
    VL_MAX_TEST      = 120,                     # cap VL eval count for time; if < N_TEST it is excluded from paired stats

    OUTPUT_DIR = "/kaggle/working",
)
if not os.path.isdir(CFG["OUTPUT_DIR"]):
    CFG["OUTPUT_DIR"] = "."        # so it also runs outside Kaggle
os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)
print("Config ready. Output dir:", CFG["OUTPUT_DIR"])

## 1. Install dependencies

Each install is wrapped so one failure does not abort the run. Make sure **Internet is ON**. For a GPU paddle build, swap `paddlepaddle` for the CUDA-matched `paddlepaddle-gpu` wheel.

In [ ]:
import subprocess, sys
def pip(*args):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)
        print("  installed:", " ".join(args)); return True
    except Exception as e:
        print("  pip FAILED:", " ".join(args), "->", e); return False

print("Installing (this can take a few minutes)...")
pip("jiwer")                              # CER / WER
pip("-U", "transformers", "accelerate")   # PaddleOCR-VL needs a recent transformers
pip("paddlepaddle")                       # CPU paddle = reliable. GPU: paddlepaddle-gpu (CUDA-matched)
pip("paddleocr")                          # PP-OCRv5
print("Done attempting installs.")

## 2. Imports, device, reproducibility, and memory/timing helpers

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter

set_seed()  # re-seed now that torch is imported
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: running on CPU — training will be slow. Enable a GPU accelerator on Kaggle.")

# Reproducible DataLoader: a fixed generator + per-worker seeding.
DATA_GEN = torch.Generator(); DATA_GEN.manual_seed(SEED)
def seed_worker(_):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

try:
    import psutil
except Exception:
    psutil = None

def gpu_mem_reset():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
def gpu_mem_peak_mb():
    return (torch.cuda.max_memory_allocated() / 1e6) if DEVICE == "cuda" else float("nan")
def rss_mb():
    return (psutil.Process().memory_info().rss / 1e6) if psutil else float("nan")

## 3. Evaluation metrics — CER and WER

`jiwer` if available, with a dependency-free Levenshtein fallback so the notebook never breaks on metrics.

In [ ]:
try:
    import jiwer
    def cer(ref, hyp): return float(jiwer.cer(ref, hyp)) if (ref and hyp) else (0.0 if ref == hyp else 1.0)
    def wer(ref, hyp): return float(jiwer.wer(ref, hyp)) if (ref and hyp) else (0.0 if ref == hyp else 1.0)
    print("Using jiwer for CER/WER.")
except Exception:
    print("jiwer unavailable -> using built-in Levenshtein.")
    def _lev(a, b):
        m, n = len(a), len(b); dp = list(range(n + 1))
        for i in range(1, m + 1):
            prev = dp[0]; dp[0] = i
            for j in range(1, n + 1):
                cur = dp[j]; dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1])); prev = cur
        return dp[n]
    def cer(ref, hyp): return _lev(list(ref), list(hyp)) / max(1, len(ref))
    def wer(ref, hyp): return _lev(ref.split(), hyp.split()) / max(1, len(ref.split()))

def eval_corpus(refs, hyps):
    pcer = [cer(r, h) for r, h in zip(refs, hyps)]
    pwer = [wer(r, h) for r, h in zip(refs, hyps)]
    return float(np.mean(pcer)), float(np.mean(pwer)), pcer

## 4. Data generation

Three renderers create line images matching the *character* of each requested dataset:
- **printed** (TRDG-like, also the base/val/test clean target domain): crisp dark text on a light page, multiple fonts.
- **scene** (SynthText proxy): text blended over a colour gradient with mild blur.
- **handwriting** (IAM proxy): slanted, wavy, blurred print.

If a real IAM / SynthText Kaggle dataset is attached it is detected and labelled `real`; otherwise the proxy is used and labelled as a proxy. Every result row records this.

In [ ]:
import glob
FONT_DIRS = ["/usr/share/fonts/truetype/dejavu", "/usr/share/fonts", "/kaggle/input"]
def find_fonts():
    fs = []
    for d in FONT_DIRS:
        fs += glob.glob(os.path.join(d, "**", "*.ttf"), recursive=True)
    fs = [f for f in fs if os.path.isfile(f)]
    return fs or [None]
FONTS = find_fonts()
print("TTF fonts found:", len(FONTS))
if FONTS == [None]:
    print("WARNING: no TTF fonts found; falling back to a tiny bitmap font (OCR will be hard for all models).")

WORDS = ("the of and to in is you that it he was for on are as with his they at be this from have "
         "or one had by word but not what all were we when your can said there use education student "
         "reading language difficult vocabulary sentence grammar comprehension passage paragraph "
         "analysis development knowledge advanced beginner complexity proficiency assessment "
         "curriculum literacy fluency context meaning structure").split()

def rand_text(nmin=2, nmax=4):
    return " ".join(random.choice(WORDS) for _ in range(random.randint(nmin, nmax)))

def _font(size):
    f = random.choice(FONTS)
    try:
        return ImageFont.truetype(f, size) if f else ImageFont.load_default()
    except Exception:
        return ImageFont.load_default()

def render_printed(text, size=None, pad=6):
    # Larger, crisp text -> learnable clean-printed domain (the CEFR target).
    size = size or random.randint(30, 36)
    fnt = _font(size); tmp = Image.new("L", (10, 10)); d = ImageDraw.Draw(tmp)
    try:
        bb = d.textbbox((0, 0), text, font=fnt); w, h = bb[2]-bb[0], bb[3]-bb[1]
    except Exception:
        w, h = len(text) * size // 2, size
    img = Image.new("L", (max(8, w) + 2*pad, max(8, h) + 2*pad), color=random.randint(245, 255))
    ImageDraw.Draw(img).text((pad, pad), text, fill=random.randint(0, 30), font=fnt)
    return img

def render_scene(text, size=None):
    base = render_printed(text, size).convert("RGB"); w, h = base.size
    bg = Image.new("RGB", (w, h)); px = bg.load()
    c1 = tuple(random.randint(0, 255) for _ in range(3)); c2 = tuple(random.randint(0, 255) for _ in range(3))
    for x in range(w):
        t = x / max(1, w - 1); col = tuple(int(c1[i]*(1-t) + c2[i]*t) for i in range(3))
        for y in range(h): px[x, y] = col
    base = base.filter(ImageFilter.GaussianBlur(0.6))
    return Image.blend(bg, base, 0.6).convert("L")

def render_handwriting_proxy(text, size=None):
    img = render_printed(text, size).rotate(random.uniform(-4, 4), expand=True, fillcolor=255)
    img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 0.9)))
    arr = np.array(img); h, w = arr.shape
    shift = (np.sin(np.linspace(0, 3*np.pi, w)) * random.uniform(0.5, 1.5)).astype(int)
    out = np.full_like(arr, 255)
    for x in range(w):
        s = int(shift[x])
        if s >= 0:
            if s < h: out[s:, x] = arr[:h-s, x]
        else:
            out[:h+s, x] = arr[-s:, x]
    return Image.fromarray(out)

print("Renderers ready.")

In [ ]:
# Detect attached real datasets (optional). Returns a directory or None.
def find_kaggle_dataset(keywords):
    root = "/kaggle/input"
    if not os.path.isdir(root):
        return None
    for d in os.listdir(root):
        if any(k in d.lower() for k in keywords):
            return os.path.join(root, d)
    return None

IAM_DIR       = find_kaggle_dataset(["iam"])
SYNTHTEXT_DIR = find_kaggle_dataset(["synthtext", "synth-text"])
print("IAM attached:      ", IAM_DIR, "(None => using the labelled handwriting proxy — this is expected, not a bug)")
print("SynthText attached:", SYNTHTEXT_DIR, "(None => using the labelled scene proxy — this is expected, not a bug)")

def make_set(kind, n):
    out = []
    for _ in range(n):
        t = rand_text()
        if   kind == "printed": img = render_printed(t)
        elif kind == "scene":   img = render_scene(t)
        elif kind == "hand":    img = render_handwriting_proxy(t)
        else:                   img = render_printed(t)
        out.append((img, t))
    return out

set_seed()  # data generation is reproducible regardless of cell run order
print("Building datasets...")
DATA = {}
DATA["base"]      = make_set("printed", CFG["N_BASE_TRAIN"])
DATA["val"]       = make_set("printed", CFG["N_VAL"])
DATA["test"]      = make_set("printed", CFG["N_TEST"])     # CEFR target domain
DATA["TRDG"]      = make_set("printed", CFG["N_FT_TRAIN"])
DATA["SynthText"] = make_set("scene",   CFG["N_FT_TRAIN"])
DATA["IAM"]       = make_set("hand",    CFG["N_FT_TRAIN"])

SOURCE = {
    "TRDG":      "synthetic(printed)",
    "SynthText": "real-subset" if SYNTHTEXT_DIR else "synthetic(scene proxy)",
    "IAM":       "real" if IAM_DIR else "synthetic(handwriting proxy)",
}
print("Dataset sizes:", {k: len(v) for k, v in DATA.items()})
print("Source tags:", SOURCE)

## 4b. Dataset verification — *is the data actually correct?*

Explicit integrity checks before any training: empty labels, out-of-charset characters, image dimensions, and **train/test label leakage**. If anything is wrong it is printed loudly here, so you never benchmark on broken data.

In [ ]:
CHARS = " " + string.ascii_letters + string.digits + ".,;:'!?-()"
CHARSET = set(CHARS)

def verify_datasets(DATA):
    rows, issues = [], []
    test_labels = set(t for _, t in DATA["test"])
    for name, samples in DATA.items():
        labels = [t for _, t in samples]
        sizes  = [im.size for im, _ in samples]
        empties = sum(1 for t in labels if len(t.strip()) == 0)
        oov = sorted({c for t in labels for c in t if c not in CHARSET})
        ws = [w for w, _ in sizes]; hs = [h for _, h in sizes]
        leak = 0
        if name in ("base", "TRDG", "SynthText", "IAM"):
            leak = sum(1 for t in labels if t in test_labels)  # exact-line overlap with the test set
        if empties: issues.append(f"{name}: {empties} EMPTY labels")
        if oov:     issues.append(f"{name}: OUT-OF-CHARSET chars {oov}")
        if min(ws) < 8 or min(hs) < 8: issues.append(f"{name}: tiny images (w>={min(ws)}, h>={min(hs)})")
        rows.append(dict(split=name, n=len(samples), empty=empties, oov_chars=len(oov),
                         leak_with_test=leak, uniq_labels=len(set(labels)),
                         w_min=min(ws), w_max=max(ws), h_min=min(hs), h_max=max(hs)))
    return pd.DataFrame(rows), issues

verify_df, issues = verify_datasets(DATA)
print(verify_df.to_string(index=False))
print()
# leakage is reported, not fatal (a few random short collisions are expected with a tiny vocab)
hard = [i for i in issues if ("EMPTY" in i or "OUT-OF-CHARSET" in i or "tiny" in i)]
if hard:
    print("DATA PROBLEMS DETECTED:")
    for i in hard: print("  -", i)
    raise AssertionError("Fix the dataset issues above before benchmarking.")
print("Dataset verification PASSED: no empty labels, all chars in charset, image sizes valid.")
print("Note: 'leak_with_test' counts exact-line overlaps with the clean test set; a few are normal")
print("      given the small word list and do not bias the comparison (all models see the same test).")
verify_df

In [ ]:
# Visual sanity check: one sample from each training distribution + the test domain.
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, key in zip(axes, ["TRDG", "SynthText", "IAM", "test"]):
    img, txt = DATA[key][0]
    ax.imshow(img, cmap="gray"); ax.set_title(f"{key}\n'{txt}'", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Persist the test crops to disk so the pre-trained engines can read them by path.
TEST_DIR = os.path.join(CFG["OUTPUT_DIR"], "test_imgs"); os.makedirs(TEST_DIR, exist_ok=True)
test_paths, test_refs = [], []
for i, (img, t) in enumerate(DATA["test"]):
    p = os.path.join(TEST_DIR, f"t{i:04d}.png")
    img.convert("RGB").save(p)
    test_paths.append(p); test_refs.append(t)
print("Saved", len(test_paths), "test images to", TEST_DIR)

## 5. Experiment 1a — pre-trained **PP-OCRv5** (traditional)

Line-level `TextRecognition` module (the recognizer stage). Wrapped so a Paddle install/runtime failure is logged and skipped, never aborting the run.

In [ ]:
def extract_rec_text(res):
    for getter in (lambda r: r["rec_text"], lambda r: r.json["rec_text"],
                   lambda r: r.json["res"]["rec_text"], lambda r: r.get("rec_text", "")):
        try:
            v = getter(res)
            if isinstance(v, str): return v
        except Exception:
            continue
    return str(res)

paddle_res = None
if CFG["RUN_PRETRAINED_PADDLEOCR"]:
    try:
        from paddleocr import TextRecognition
        try:
            rec = TextRecognition(model_name=CFG["PADDLE_REC_MODEL"], device=CFG["PADDLE_DEVICE"])
        except Exception:
            try: rec = TextRecognition(model_name=CFG["PADDLE_REC_MODEL"])
            except Exception: rec = TextRecognition()
        hyps = []; t0 = time.time()
        for p in test_paths:
            out = rec.predict(p); txt = ""
            for r in (out if isinstance(out, list) else [out]):
                txt = extract_rec_text(r)
            hyps.append(txt)
        dt = (time.time() - t0) / max(1, len(test_paths))
        mc, mw, pc = eval_corpus(test_refs, hyps)
        paddle_res = dict(model="PP-OCRv5 (pretrained)", CER=mc, WER=mw, sec_per_img=dt,
                          device=CFG["PADDLE_DEVICE"], mem_mb=rss_mb(), source="real", per_cer=pc)
        print(f"PP-OCRv5  CER={mc:.4f}  WER={mw:.4f}  {dt*1000:.1f} ms/img  ({CFG['PADDLE_DEVICE']})")
    except Exception as e:
        print("PaddleOCR eval skipped/failed (continuing):", repr(e))

## 5. Experiment 1b — pre-trained **PaddleOCR-VL** (vision-language)

Element-level recognition via `transformers`. Capped at `VL_MAX_TEST` images for runtime; if that is below `N_TEST` the VL row is automatically excluded from the *paired* statistics (it would not share the full test set).

In [ ]:
vl_res = None
if CFG["RUN_PRETRAINED_VL"]:
    try:
        from transformers import AutoModelForCausalLM, AutoProcessor
        try:
            vl = AutoModelForCausalLM.from_pretrained(CFG["VL_MODEL_ID"], trust_remote_code=True, dtype=torch.bfloat16)
        except TypeError:
            vl = AutoModelForCausalLM.from_pretrained(CFG["VL_MODEL_ID"], trust_remote_code=True, torch_dtype=torch.bfloat16)
        vl = vl.to(DEVICE).eval()
        proc = AutoProcessor.from_pretrained(CFG["VL_MODEL_ID"], trust_remote_code=True)

        n = min(CFG["VL_MAX_TEST"], len(DATA["test"]))
        refs_vl = test_refs[:n]
        gpu_mem_reset(); hyps = []; t0 = time.time()
        for img, _t in DATA["test"][:n]:
            messages = [{"role": "user", "content": "OCR"}]
            prompt = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inp = proc(text=[prompt], images=[img.convert("RGB")], return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                g = vl.generate(**inp, max_new_tokens=64, do_sample=False)
            gen = g[:, inp["input_ids"].shape[1]:]
            hyps.append(proc.batch_decode(gen, skip_special_tokens=True)[0].strip())
        dt = (time.time() - t0) / max(1, n)
        mc, mw, pc = eval_corpus(refs_vl, hyps)
        vl_res = dict(model="PaddleOCR-VL (pretrained)", CER=mc, WER=mw, sec_per_img=dt,
                      device=DEVICE, mem_mb=gpu_mem_peak_mb(), source="real", per_cer=pc)
        print(f"PaddleOCR-VL  CER={mc:.4f}  WER={mw:.4f}  {dt*1000:.1f} ms/img  (n={n}, {DEVICE})")
        del vl; gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
    except Exception as e:
        print("PaddleOCR-VL eval skipped/failed (continuing):", repr(e))

## 6. The trainable recognizer (CRNN + CTC)

A canonical **CNN (64→128→256→256→512) → BiLSTM → CTC** recognizer (Shi et al. widths) — the faithful, reproducible stand-in for a traditional OCR recognition stage.

Two design choices make it **actually train** (without them CTC collapses to all-blank and every model ties — the classic silent failure):
- **Aspect-preserving preprocessing**: resize to height `IMG_H`, then pad width to `IMG_W` (no horizontal stretching that destroys glyphs).
- **Blank-bias init** (`fc.bias[0] = -3`) plus the **LR warmup** in the trainer, which together keep the model out of the all-blank minimum in the first epochs.

In [ ]:
char2idx = {c: i + 1 for i, c in enumerate(CHARS)}   # index 0 reserved for CTC blank
idx2char = {i + 1: c for i, c in enumerate(CHARS)}
NUM_CLASSES = len(CHARS) + 1

def encode(text):
    return [char2idx[c] for c in text if c in char2idx]

def to_tensor(img):
    # Aspect-preserving: scale to height IMG_H, pad (or truncate) width to IMG_W.
    img = img.convert("L"); w, h = img.size
    nw = min(CFG["IMG_W"], max(1, int(round(w * CFG["IMG_H"] / h))))
    img = img.resize((nw, CFG["IMG_H"]))
    canvas = Image.new("L", (CFG["IMG_W"], CFG["IMG_H"]), color=250)  # light pad = background
    canvas.paste(img, (0, 0))
    a = (np.asarray(canvas, dtype=np.float32) / 255.0 - 0.5) / 0.5
    return torch.from_numpy(a).unsqueeze(0)   # (1, H, W)

class OCRDS(Dataset):
    def __init__(self, samples): self.s = samples
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        img, txt = self.s[i]; return to_tensor(img), txt

def collate(batch):
    imgs = torch.stack([b[0] for b in batch])
    texts = [b[1] for b in batch]
    targs = [torch.tensor(encode(t), dtype=torch.long) for t in texts]
    tlens = torch.tensor([len(t) for t in targs], dtype=torch.long)
    targs = torch.cat(targs) if len(targs) else torch.zeros(0, dtype=torch.long)
    return imgs, targs, tlens, texts

class CRNN(nn.Module):
    def __init__(self, nclass, blank_bias=-3.0):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),                                  # H/2,  W/2
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),                                # H/4,  W/4
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(), nn.MaxPool2d((2, 1), (2, 1)),                     # H/8,  W/4
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d((2, 1), (2, 1)),# H/16, W/4
            nn.Conv2d(512, 512, 2, 1, 0), nn.ReLU(),                                                   # H->1
        )
        self.rnn = nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(512, nclass)
        nn.init.constant_(self.fc.bias[0], blank_bias)   # discourage the all-blank plateau early
    def forward(self, x):
        c = self.cnn(x)                       # (B, 512, 1, W')
        b, ch, hh, ww = c.size()
        assert hh == 1, f"feature height is {hh}, expected 1 (check IMG_H={CFG['IMG_H']})"
        c = c.squeeze(2).permute(0, 2, 1)     # (B, W', 512)
        r, _ = self.rnn(c)
        return self.fc(r)                     # (B, T, nclass)

def greedy_decode(logits):
    idx = logits.argmax(-1).cpu().numpy(); outs = []
    for seq in idx:
        prev, chars = 0, []
        for s in seq:
            s = int(s)
            if s != 0 and s != prev: chars.append(idx2char.get(s, ""))
            prev = s
        outs.append("".join(chars))
    return outs

print("CRNN defined. NUM_CLASSES =", NUM_CLASSES)

In [ ]:
def run_eval(model, samples):
    model.eval()
    dl = DataLoader(OCRDS(samples), batch_size=CFG["BATCH"], collate_fn=collate)
    refs, hyps = [], []
    gpu_mem_reset(); t0 = time.time()
    with torch.no_grad():
        for imgs, _t, _l, texts in dl:
            imgs = imgs.to(DEVICE)
            hyps += greedy_decode(model(imgs)); refs += texts
    dt = (time.time() - t0) / max(1, len(samples))
    mc, mw, pc = eval_corpus(refs, hyps)
    return mc, mw, pc, dt, gpu_mem_peak_mb()

def train_model(model, train_samples, val_samples, epochs, tag=""):
    """Train with LR warmup + cosine decay, and RESTORE the best-val-CER epoch
    (not the last). This is what guarantees you keep the genuinely best model."""
    set_seed()                          # deterministic training -> stable, repeatable results
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"])
    ctc = nn.CTCLoss(blank=0, zero_infinity=True)
    dl = DataLoader(OCRDS(train_samples), batch_size=CFG["BATCH"], shuffle=True,
                    collate_fn=collate, generator=DATA_GEN, worker_init_fn=seed_worker)
    warm = max(1, CFG["WARMUP_EPOCHS"])
    hist = dict(train_loss=[], val_cer=[])
    best_cer, best_state, best_ep = float("inf"), None, -1
    for ep in range(epochs):
        if ep < warm:
            lr = CFG["LR"] * (ep + 1) / warm                                   # linear warmup
        else:
            prog = (ep - warm) / max(1, epochs - warm)
            lr = 0.5 * CFG["LR"] * (1 + math.cos(math.pi * prog))              # cosine decay
        for g in opt.param_groups: g["lr"] = lr
        model.train(); tot, nb = 0.0, 0
        for imgs, targets, tlens, _texts in dl:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            log = model(imgs)
            logp = log.log_softmax(2).permute(1, 0, 2)                         # (T, B, C)
            T = logp.size(0)
            ilens = torch.full((imgs.size(0),), T, dtype=torch.long)
            loss = ctc(logp, targets, ilens, tlens)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            tot += float(loss.detach()); nb += 1
        vc, _, _, _, _ = run_eval(model, val_samples)
        hist["train_loss"].append(tot / max(1, nb)); hist["val_cer"].append(vc)
        if vc < best_cer - 1e-6:
            best_cer, best_ep = vc, ep + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  [{tag}] epoch {ep+1}/{epochs}  loss={tot/max(1,nb):.3f}  val_CER={vc:.3f}"
              + ("  <- best" if best_ep == ep + 1 else ""))
    if best_state is not None:
        model.load_state_dict(best_state)                                      # keep the BEST epoch
    hist["best_val_cer"], hist["best_epoch"] = best_cer, best_ep
    print(f"  [{tag}] -> restored BEST epoch {best_ep} (val_CER={best_cer:.4f})")
    return model, hist

print("Training utilities ready (warmup + best-checkpoint restore).")

## 7. Run Experiments 1–6

Train the base ("pre-trained") CRNN on the clean printed corpus, then fine-tune deep copies on each dataset configuration. A **convergence sanity-gate** runs right after the base model: if it did not learn, the whole benchmark is flagged untrustworthy.

In [ ]:
import copy

results = []          # one dict per model -> comparison table
curves  = {}          # tag -> training history
percer  = {}          # tag -> per-sample CER on the test set (for statistics)
trained_models = {}   # tag -> trained CRNN (kept so the best one can be saved/downloaded)

# Register the pre-trained anchors (if they ran).
for r in (paddle_res, vl_res):
    if r:
        results.append({k: r[k] for k in ["model", "CER", "WER", "sec_per_img", "device", "mem_mb", "source"]})
        percer[r["model"]] = r["per_cer"]

if CFG["RUN_FINETUNE_EXPERIMENTS"]:
    print("Training base CRNN (the 'pre-trained' recognizer)...")
    base, h = train_model(CRNN(NUM_CLASSES), DATA["base"], DATA["val"], CFG["EPOCHS_BASE"], tag="CRNN-base")
    curves["CRNN base (no FT)"] = h
    mc, mw, pc, dt, mem = run_eval(base, DATA["test"])
    results.append(dict(model="CRNN base (no FT)", CER=mc, WER=mw, sec_per_img=dt,
                        device=DEVICE, mem_mb=mem, source="trained"))
    percer["CRNN base (no FT)"] = pc
    trained_models["CRNN base (no FT)"] = base
    print(f"  -> base test CER={mc:.4f} WER={mw:.4f}")

    # ---- CONVERGENCE SANITY-GATE ----
    base_val = curves["CRNN base (no FT)"]["best_val_cer"]
    print("\n" + "="*70)
    if base_val > CFG["SANITY_CER_THRESHOLD"]:
        print("!! WARNING — the base recognizer did NOT converge "
              f"(best val_CER={base_val:.3f} > {CFG['SANITY_CER_THRESHOLD']}).")
        print("!! The CRNN rows below are NOT trustworthy. To fix: ensure GPU is ON, and")
        print("!! increase EPOCHS_BASE / EPOCHS_FT / N_BASE_TRAIN, then re-run from Section 0.")
    else:
        print(f"OK — sanity check PASSED: base recognizer converged (best val_CER={base_val:.3f}).")
    print("="*70 + "\n")

    def finetune(name, train_samples, epochs):
        print(f"Fine-tuning -> {name}")
        m = copy.deepcopy(base)
        m, hh = train_model(m, train_samples, DATA["val"], epochs, tag=name)
        curves[name] = hh
        mc, mw, pc, dt, mem = run_eval(m, DATA["test"])
        results.append(dict(model=name, CER=mc, WER=mw, sec_per_img=dt,
                            device=DEVICE, mem_mb=mem, source="trained"))
        percer[name] = pc
        trained_models[name] = m
        print(f"  -> {name} test CER={mc:.4f} WER={mw:.4f}")
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()

    finetune("FT: TRDG",            DATA["TRDG"],                                   CFG["EPOCHS_FT"])
    finetune("FT: SynthText",       DATA["SynthText"],                             CFG["EPOCHS_FT"])
    if CFG["INCLUDE_IAM_EXPERIMENT"]:
        finetune("FT: IAM",         DATA["IAM"],                                   CFG["EPOCHS_FT"])
    finetune("FT: TRDG+SynthText",  DATA["TRDG"] + DATA["SynthText"],              CFG["EPOCHS_FT"])
    if CFG["INCLUDE_IAM_EXPERIMENT"]:
        finetune("FT: TRDG+SynthText+IAM", DATA["TRDG"] + DATA["SynthText"] + DATA["IAM"], CFG["EPOCHS_FT"])

print("\nAll experiments complete. Models evaluated:", len(results))

## 8. Results, plots, statistics, and final ranking

### 8.1 Master comparison table

In [ ]:
df = pd.DataFrame(results).sort_values("CER").reset_index(drop=True)
df_disp = df.copy()
for col, nd in [("CER", 4), ("WER", 4), ("sec_per_img", 4), ("mem_mb", 1)]:
    if col in df_disp: df_disp[col] = df_disp[col].round(nd)
df_disp.to_csv(os.path.join(CFG["OUTPUT_DIR"], "ocr_benchmark_results.csv"), index=False)
print("Saved ocr_benchmark_results.csv")
df_disp

### 8.2 CER / WER comparison plots

In [ ]:
d2 = df.sort_values("CER")
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].barh(d2["model"], d2["CER"]);                  ax[0].set_title("CER (lower = better)"); ax[0].invert_yaxis()
ax[1].barh(d2["model"], d2["WER"], color="orange");  ax[1].set_title("WER (lower = better)"); ax[1].invert_yaxis()
for a in ax: a.set_xlabel("error rate")
plt.tight_layout(); plt.savefig(os.path.join(CFG["OUTPUT_DIR"], "cer_wer.png"), dpi=120); plt.show()

### 8.3 Training curves

In [ ]:
if curves:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for name, hh in curves.items():
        xs = range(1, len(hh["train_loss"]) + 1)
        ax[0].plot(xs, hh["train_loss"], marker="o", label=name)
        ax[1].plot(xs, hh["val_cer"], marker="o", label=name)
    ax[0].set_title("Training loss");  ax[0].set_xlabel("epoch"); ax[0].set_ylabel("CTC loss"); ax[0].legend(fontsize=8)
    ax[1].set_title("Validation CER"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("CER");      ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(CFG["OUTPUT_DIR"], "training_curves.png"), dpi=120); plt.show()
else:
    print("No training curves (fine-tuning experiments disabled).")

### 8.4 Statistical analysis of improvements

For every model sharing the test set with the **CRNN base** baseline: mean per-image CER change, a 95% bootstrap CI, and a Wilcoxon signed-rank p-value (NaN-safe when there is no difference).

In [ ]:
from scipy import stats
set_seed()  # makes the bootstrap reproducible
base_key = "CRNN base (no FT)"
rows = []
if base_key in percer:
    b = np.array(percer[base_key])
    for name, pc in percer.items():
        if name == base_key: continue
        pc = np.array(pc)
        if len(pc) != len(b): continue          # only paired comparisons on identical test images
        diffs = pc - b
        delta = float(diffs.mean())             # negative => lower CER => improvement
        if np.allclose(diffs, 0):
            p = 1.0
        else:
            try: _w, p = stats.wilcoxon(pc, b)
            except Exception: p = float("nan")
        boot = [float(np.mean(np.random.choice(diffs, len(diffs), replace=True))) for _ in range(2000)]
        lo, hi = np.percentile(boot, [2.5, 97.5])
        verdict = ("IMPROVED" if (delta < 0 and p < 0.05) else
                   "WORSE"    if (delta > 0 and p < 0.05) else "n.s.")
        rows.append(dict(model=name, mean_CER_delta=round(delta, 4),
                         CI95_low=round(float(lo), 4), CI95_high=round(float(hi), 4),
                         p_value=round(float(p), 5), verdict=verdict))
    stats_df = pd.DataFrame(rows).sort_values("mean_CER_delta").reset_index(drop=True)
    stats_df.to_csv(os.path.join(CFG["OUTPUT_DIR"], "stats_analysis.csv"), index=False)
    print("vs. baseline:", base_key, "(negative delta = improvement)")
else:
    stats_df = pd.DataFrame(); print("Baseline CRNN not available -> statistics skipped.")
stats_df

### 8.5 Final ranking and best-approach decision

In [ ]:
ranked = df.sort_values("CER").reset_index(drop=True)
print("=" * 74)
print("FINAL RANKING - by CER on clean printed English (the CEFR target domain)")
print("=" * 74)
for i, row in ranked.iterrows():
    spi = row["sec_per_img"]
    print(f"{i+1:>2}. {row['model']:<30} CER={row['CER']:.4f}  WER={row['WER']:.4f}  "
          f"{spi*1000:7.1f} ms/img  [{row['device']}/{row['source']}]")

best = ranked.iloc[0]
print("\n" + "-" * 74)
print(f"BEST OCR APPROACH for CEFR text extraction: {best['model']}")
print(f"   CER={best['CER']:.4f}  WER={best['WER']:.4f}  ({best['sec_per_img']*1000:.1f} ms/img)")
print("-" * 74)

### How to read these results (for the CEFR goal)
- The **target domain is clean printed English**, so the ranking is by **CER on the clean printed test set**.
- A strong pre-trained model (PP-OCRv5 / PaddleOCR-VL) is expected to lead on clean print. The fine-tuning study answers a different question: **does fine-tuning a traditional recognizer on synthetic / scene / handwriting data help or hurt on the clean target?** Often TRDG (printed) helps while scene/handwriting data can *hurt* the clean-print CER — a useful, honest finding.
- Trust the CRNN rows only if the **sanity-gate passed**. If it warned, increase the epoch/data budget and re-run.

## 9. Save the best model for download (Kaggle)

Saves the best **trainable** recognizer to `best_recognizer.pt`, writes `best_model_info.json`, and **zips every output** into `ocr_benchmark_outputs.zip` so you can download one file from the Kaggle *Output* tab.

In [ ]:
import json, zipfile
ranked = df.sort_values("CER").reset_index(drop=True)
best_name = ranked.iloc[0]["model"]; best_row = ranked.iloc[0].to_dict()
tm = globals().get("trained_models", {})
save_dir = CFG["OUTPUT_DIR"]; saved_paths = []
print("Best model overall by CER:", best_name)

# 1) Always save the best TRAINABLE (CRNN) checkpoint, restored to its best epoch.
trainable_in_rank = [m for m in ranked["model"].tolist() if m in tm]
if trainable_in_rank:
    best_crnn_name = trainable_in_rank[0]
    best_crnn = tm[best_crnn_name].to("cpu")
    ckpt = dict(state_dict=best_crnn.state_dict(), chars=CHARS, num_classes=NUM_CLASSES,
                img_h=CFG["IMG_H"], img_w=CFG["IMG_W"],
                arch="CRNN cnn[64-128-256-256-512]+BiLSTM256+CTC(blank=0)",
                experiment=best_crnn_name,
                test_CER=float(ranked.set_index("model").loc[best_crnn_name, "CER"]),
                best_val_CER=float(curves.get(best_crnn_name, {}).get("best_val_cer", float("nan"))))
    p = os.path.join(save_dir, "best_recognizer.pt"); torch.save(ckpt, p); saved_paths.append(p)
    print("Saved best trainable recognizer:", best_crnn_name, "->", p)
else:
    print("No trainable models retained (fine-tuning was disabled).")

# 2) Record the overall best (may be a pretrained model not held as a checkpoint).
info = dict(best_overall_model=best_name,
            best_overall_metrics={k: best_row[k] for k in ["CER","WER","sec_per_img","device","source"] if k in best_row},
            how_to_load={
                "PP-OCRv5 (pretrained)": f"from paddleocr import TextRecognition; TextRecognition(model_name='{CFG['PADDLE_REC_MODEL']}')",
                "PaddleOCR-VL (pretrained)": f"AutoModelForCausalLM.from_pretrained('{CFG['VL_MODEL_ID']}', trust_remote_code=True)",
                "CRNN checkpoint": "load best_recognizer.pt with load_recognizer() (next cell)"})
ip = os.path.join(save_dir, "best_model_info.json")
with open(ip, "w") as f: json.dump(info, f, indent=2)
saved_paths.append(ip); print("Saved best_model_info.json")

# 3) Collect csv/png artefacts and zip everything for a one-click download.
for extra in ["ocr_benchmark_results.csv", "stats_analysis.csv", "cer_wer.png", "training_curves.png"]:
    pth = os.path.join(save_dir, extra)
    if os.path.exists(pth) and pth not in saved_paths: saved_paths.append(pth)
zip_path = os.path.join(save_dir, "ocr_benchmark_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in saved_paths:
        if os.path.exists(p): z.write(p, arcname=os.path.basename(p))
print("\nZipped all outputs ->", zip_path)
print("Download from the Kaggle 'Output' tab (right panel). Files:")
for p in saved_paths + [zip_path]: print("  -", p)

In [ ]:
# Reload helper: rebuild the saved recognizer for inference in a fresh session.
def load_recognizer(ckpt_path, device="cpu"):
    ck = torch.load(ckpt_path, map_location=device)
    model = CRNN(ck["num_classes"]).to(device)
    model.load_state_dict(ck["state_dict"]); model.eval()
    return model, ck

def recognize(model, pil_image, device="cpu"):
    with torch.no_grad():
        return greedy_decode(model(to_tensor(pil_image).unsqueeze(0).to(device)))[0]

# Example (uncomment after a run):
# m, ck = load_recognizer(os.path.join(CFG["OUTPUT_DIR"], "best_recognizer.pt"), device=DEVICE)
# print("pred:", recognize(m, DATA["test"][0][0], device=DEVICE), "| ref:", DATA["test"][0][1])
print("Reload helpers ready: load_recognizer(), recognize().")

---
## Appendix A — Fine-tuning the **real** PP-OCRv5 recognizer (official recipe)

The legitimate way to fine-tune the actual PP-OCRv5 weights (kept out of the auto-run path: it needs the PaddleOCR repo + a CUDA-matched paddle wheel and more time). PaddleOCR expects a label file of `image_path<TAB>label` lines.

```bash
pip install paddlepaddle-gpu          # wheel matching your CUDA
git clone https://github.com/PaddlePaddle/PaddleOCR && cd PaddleOCR && pip install -e .
```
```python
def dump_rec_labels(samples, img_dir, label_file):
    import os
    os.makedirs(img_dir, exist_ok=True)
    with open(label_file, "w") as f:
        for i, (img, txt) in enumerate(samples):
            p = os.path.join(img_dir, f"{i:06d}.png"); img.convert("RGB").save(p)
            f.write(f"{p}\t{txt}\n")
# dump_rec_labels(DATA["TRDG"], "rec_train_imgs", "rec_train.txt")
# dump_rec_labels(DATA["val"],  "rec_val_imgs",   "rec_val.txt")
```
```bash
python tools/train.py -c configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml \
    -o Global.pretrained_model=./pretrain/PP-OCRv5_mobile_rec \
       Train.dataset.label_file_list=["rec_train.txt"] \
       Eval.dataset.label_file_list=["rec_val.txt"] Global.epoch_num=20
```
Then point Section 5's `TextRecognition(model_name=..., model_dir=...)` at the trained weights to benchmark fine-tuned PP-OCRv5 against everything above.

## Appendix B — PaddleOCR-VL LoRA fine-tuning (experimental)

The official PaddleOCR-VL fine-tuning workflow (ERNIEKit) was **not yet released** at time of writing; treat this as an experimental scaffold. It applies PEFT/LoRA so the VLM fits a single GPU.

```python
# pip install peft
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
model = AutoModelForCausalLM.from_pretrained(CFG["VL_MODEL_ID"], trust_remote_code=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj"])  # adjust to the VL arch
model = get_peft_model(model, lora); model.print_trainable_parameters()
# Build (image, "OCR" prompt, target_text) batches with the processor and train with the
# standard causal-LM loss. Use bf16 + gradient checkpointing to fit a T4.
```
Until the official recipe lands, the honest move for a CEFR pipeline is to use PaddleOCR-VL **pre-trained**.

## Appendix C — Plugging the winning OCR into the CEFR regressor

```python
def document_text_to_cefr(image_or_pil, ocr_fn, cefr_model, tokenizer, device="cuda"):
    text = ocr_fn(image_or_pil)                       # the best model from the ranking above
    enc = tokenizer(text, truncation=True, max_length=512, return_tensors="pt").to(device)
    import torch
    with torch.no_grad():
        score = cefr_model(enc["input_ids"], enc["attention_mask"]).item()
    return text, score   # continuous A1..C2 (1..6) scale
```
Evaluate **end-to-end**: compare CEFR MAE / RMSE / QWK using ground-truth text vs OCR text. If the gap is small, your OCR is good enough — stop optimizing it.

## Appendix D — Swapping in the real datasets

- **Real TRDG:** `pip install trdg`, then `from trdg.generators import GeneratorFromStrings`; wrap each `(PIL_image, text)` pair into `DATA["TRDG"]`.
- **Real SynthText:** attach a SynthText (subset) Kaggle dataset; `SYNTHTEXT_DIR` is auto-detected. Parse `gt.mat` to crop word images + labels and replace `DATA["SynthText"]`.
- **Real IAM:** attach an IAM Kaggle dataset (registration required). `IAM_DIR` is auto-detected; parse the `lines`/`words` images + ASCII ground-truth and replace `DATA["IAM"]`.

The `source` column in every results table records **real** vs **labelled proxy**, so your conclusions stay honest. After swapping data, re-run from Section 0 — seeding keeps everything reproducible.